In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq

In [ ]:
def lpc(x: np.ndarray, K: int) -> np.ndarray:
    """The coefficients h_1, ..., h_K that best predict each x[n] from x[n-1], ..., x[n-K]."""
    N = len(x)
    X = np.stack([x[K - k:N - k] for k in range(1, K + 1)], axis=1)   # row n: x[n-1], ..., x[n-K]
    h, *_ = np.linalg.lstsq(X, x[K:], rcond=None)                     # minimize the squared error
    return h


vowel = pq.Audio.from_file("../assets/vowel-a.wav")
f_s = vowel.sample_rate
x = vowel.segment(offset=1.0, duration=0.5).samples[:, 0].astype(np.float64)
K = 40
h = lpc(x, K)

# Predict every sample from the K samples before it
X = np.stack([x[K - k:len(x) - k] for k in range(1, K + 1)], axis=1)
prediction = X @ h
error = x[K:] - prediction
print(f"The prediction error is {10 * np.log10(np.sum(x[K:] ** 2) / np.sum(error ** 2)):.1f} dB "
      "quieter than the vowel")

t = 1000 * np.arange(K, K + 1000) / f_s
plt.figure(figsize=(10, 3.5))
plt.plot(t, x[K:K + 1000], label="$x[n]$")
plt.plot(t, prediction[:1000], "--", label=r"$\hat{x}[n]$")
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.legend(loc="upper right")
plt.show()
pq.play(vowel)